In [1]:
import tensorquant as tq
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Automatic Adjoint Differentiation (AAD) with TensorFlow

This section introduces **Automatic Adjoint Differentiation (AAD)** using TensorFlow. AAD is a computational technique that allows automatic and efficient calculation of derivatives (gradients) of complex functions by leveraging the chain rule.

## Why AAD?

- **Efficiency**: Computes all partial derivatives in a single pass
- **Precision**: Avoids numerical approximation errors
- **Scalability**: Works well even with very complex functions
- **Integration**: TensorFlow provides native AAD through `GradientTape`

## How it works?

TensorFlow automatically traces all operations performed on trackable variables (like `tf.Variable`) and builds a computational graph. When we request the gradient, TensorFlow performs a **backward pass** applying the chain rule to calculate all partial derivatives.

## Chain rule and the tape

- **Chain rule**: if we have a composition of functions $y = f(g(h(x)))$, the derivative is
  $$\frac{dy}{dx} = f'(g(h(x))) \cdot g'(h(x)) \cdot h'(x).$$
  In AAD, this is implemented by propagating derivatives backward through each elementary operation in the reverse order in which they were applied.
- **Tape as an operation log**: `tf.GradientTape` records (on a *tape*) every operation applied to the watched variables: additions, multiplications, exponentials, etc.
- **Backward sweep**: when we call `tape.gradient(y, x)`, TensorFlow reads this tape in reverse, applies the chain rule step-by-step, and accumulates the contributions to the gradient of each intermediate operation.
- **Multiple inputs / outputs**: the same tape can be used to compute derivatives of one output with respect to many inputs, which is exactly what we need for Greeks in quantitative finance.

## Relation to backpropagation

In deep learning, **backpropagation** is just reverse-mode automatic differentiation applied to a neural network.

- A neural network is a composition of many simple functions (layers, activations, normalizations, etc.).
- During the **forward pass**, we compute the output and record all operations on a tape (or computation graph).
- During the **backward pass**, we apply the **same chain rule** as in AAD, propagating gradients from the output back to each parameter.
- The gradient of the loss w.r.t. millions of parameters is computed in one backward sweep, exactly like computing many Greeks from a single pricing function.


In [2]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)
print("\n=== Example 1: Simple function f(x) = x^2 ===\n")

# Create a trackable variable
x = tf.Variable(3.0, dtype=tf.float64)

# Use GradientTape to trace operations
with tf.GradientTape() as tape:
    y = x ** 2  # f(x) = x^2

# Calculate the derivative: df/dx = 2x
dy_dx = tape.gradient(y, x)
print(f"f(x) = x^2, with x = {x.numpy()}")
print(f"f(x) = {y.numpy()}")
print(f"df/dx = {dy_dx.numpy()} (expected: {2 * x.numpy()})")


TensorFlow version: 2.20.0

=== Example 1: Simple function f(x) = x^2 ===

f(x) = x^2, with x = 3.0
f(x) = 9.0
df/dx = 6.0 (expected: 6.0)


In [3]:
print("=== Example 2: Multivariate function f(x, y) = x^2 + y^2 + xy ===\n")

x = tf.Variable(2.0, dtype=tf.float64)
y = tf.Variable(3.0, dtype=tf.float64)

with tf.GradientTape() as tape:
    f = x**2 + y**2 + x * y

# Calculate gradients with respect to both variables
gradients = tape.gradient(f, [x, y])
df_dx, df_dy = gradients

print(f"f(x, y) = x^2 + y^2 + xy, with x = {x.numpy()}, y = {y.numpy()}")
print(f"f(x, y) = {f.numpy()}")
print(f"\nGradients:")
print(f"∂f/∂x = {df_dx.numpy()} (expected: {2*x.numpy() + y.numpy()})")
print(f"∂f/∂y = {df_dy.numpy()} (expected: {2*y.numpy() + x.numpy()})")


=== Example 2: Multivariate function f(x, y) = x^2 + y^2 + xy ===

f(x, y) = x^2 + y^2 + xy, with x = 2.0, y = 3.0
f(x, y) = 19.0

Gradients:
∂f/∂x = 7.0 (expected: 7.0)
∂f/∂y = 8.0 (expected: 8.0)


In [4]:
print("=== Example 3: Composite function with transcendental functions ===\n")
print("f(x) = exp(x^2) * sin(x)\n")

x = tf.Variable(1.0, dtype=tf.float64)

with tf.GradientTape() as tape:
    f = tf.exp(x**2) * tf.sin(x)

df_dx = tape.gradient(f, x)

# Verify with analytical derivative: d/dx[exp(x^2) * sin(x)]
# = exp(x^2) * 2x * sin(x) + exp(x^2) * cos(x)
# = exp(x^2) * (2x * sin(x) + cos(x))
expected = tf.exp(x**2) * (2*x*tf.sin(x) + tf.cos(x))

print(f"x = {x.numpy()}")
print(f"f(x) = {f.numpy()}")
print(f"df/dx (AAD) = {df_dx.numpy()}")
print(f"df/dx (analytical) = {expected.numpy()}")
print(f"Error: {abs(df_dx.numpy() - expected.numpy())}")


=== Example 3: Composite function with transcendental functions ===

f(x) = exp(x^2) * sin(x)

x = 1.0
f(x) = 2.2873552871788423
df/dx (AAD) = 6.04340451427357
df/dx (analytical) = 6.04340451427357
Error: 0.0


In [5]:
print("=== Example 4: Function with vector operations (similar to Monte Carlo) ===\n")
print("f(x) = mean(exp(x * z)) where z is a vector of random values\n")

# Simulate a Monte Carlo-like case
n_samples = 10000
x = tf.Variable(0.5, dtype=tf.float64)
z = tf.random.normal((n_samples,), seed=42, dtype=tf.float64)

with tf.GradientTape() as tape:
    # Calculate exp(x * z) for each sample
    exp_values = tf.exp(x * z)
    # Mean over all samples
    f = tf.reduce_mean(exp_values)

df_dx = tape.gradient(f, x)

# Analytical derivative: d/dx[mean(exp(x*z))] = mean(z * exp(x*z))
expected = tf.reduce_mean(z * tf.exp(x * z))

print(f"x = {x.numpy()}")
print(f"f(x) = {f.numpy()}")
print(f"df/dx (AAD) = {df_dx.numpy()}")
print(f"df/dx (analytical) = {expected.numpy()}")
print(f"Error: {abs(df_dx.numpy() - expected.numpy())}")


=== Example 4: Function with vector operations (similar to Monte Carlo) ===

f(x) = mean(exp(x * z)) where z is a vector of random values

x = 0.5
f(x) = 1.1369896655556062
df/dx (AAD) = 0.5758864727906581
df/dx (analytical) = 0.5758864727906581
Error: 0.0


In [6]:
print("=== Example 5: Multiple gradients computation (Greeks) ===\n")
print("Simulate a function that depends on multiple parameters\n")
print("f(S, r, sigma, T) = S * exp((r - 0.5*sigma^2)*T + sigma*sqrt(T)*z)\n")

# Parameters (similar to Black-Scholes)
S = tf.Variable(100.0, dtype=tf.float64)  # Spot
r = tf.Variable(0.05, dtype=tf.float64)  # Risk-free rate
sigma = tf.Variable(0.2, dtype=tf.float64)  # Volatility
T = tf.Variable(1.0, dtype=tf.float64)  # Time
z = tf.constant(0.0, dtype=tf.float64)  # Standardized shock
strike = tf.constant(100., dtype=tf.float64) 

with tf.GradientTape(persistent=True) as tape:
    # Calculate future value
    drift = (r - 0.5 * sigma**2) * T
    diffusion = sigma * tf.sqrt(T) * z
    future_value = S * tf.exp(drift + diffusion)
    future_value = future_value - strike

# Calculate all gradients (Greeks)
dS = tape.gradient(future_value, S)  # Delta-like
dr = tape.gradient(future_value, r)  # Rho-like
dsigma = tape.gradient(future_value, sigma)  # Vega-like
dT = tape.gradient(future_value, T)  # Theta-like

print(f"Parameters: S={S.numpy()}, r={r.numpy()}, σ={sigma.numpy()}, T={T.numpy()}")
print(f"Future Value = {future_value.numpy()}\n")
print("Gradients (Greeks):")
print(f"∂f/∂S (Delta-like) = {dS.numpy()}")
print(f"∂f/∂r (Rho-like) = {dr.numpy()}")
print(f"∂f/∂σ (Vega-like) = {dsigma.numpy()}")
print(f"∂f/∂T (Theta-like) = {dT.numpy()}")


=== Example 5: Multiple gradients computation (Greeks) ===

Simulate a function that depends on multiple parameters

f(S, r, sigma, T) = S * exp((r - 0.5*sigma^2)*T + sigma*sqrt(T)*z)

Parameters: S=100.0, r=0.05, σ=0.2, T=1.0
Future Value = 3.0454533953517

Gradients (Greeks):
∂f/∂S (Delta-like) = 1.030454533953517
∂f/∂r (Rho-like) = 103.0454533953517
∂f/∂σ (Vega-like) = -20.60909067907034
∂f/∂T (Theta-like) = 3.091363601860551




## Forward Pass and Backward Pass on a Composite Function

Consider a composite function $y = f(g(h(x)))$ built from three elementary stages:

$$h(x) = x^2 + 1, \qquad g(u) = e^u, \qquad f(v) = \sin(v)$$

### Forward Pass

The forward pass computes the function value by propagating the input towards the output, **storing the intermediate values** (which will be needed during the backward pass):

| Step | Operation | Value (at $x=1$) |
|------|-----------|-----------------|
| 1 | $u = h(x) = x^2 + 1$ | $u = 2$ |
| 2 | $v = g(u) = e^u$ | $v = e^2 \approx 7.389$ |
| 3 | $y = f(v) = \sin(v)$ | $y = \sin(e^2) \approx 0.894$ |

### Backward Pass (chain rule)

The backward pass starts from the output seed ($\bar{y} = 1$) and propagates it **in reverse**, multiplying by the local derivative of each node:

$$\frac{dy}{dx} = \underbrace{f'(v)}_{\cos(v)} \cdot \underbrace{g'(u)}_{e^u} \cdot \underbrace{h'(x)}_{2x}$$

In `tf.GradientTape` this happens automatically: the tape records all operations during the forward pass, then `tape.gradient()` executes the backward sweep.



In [7]:

# ── Define the three elementary functions ─────────────────────────────────────
def h(x):
    """h(x) = x² + 1"""
    return x**2 + 1.0

def g(u):
    """g(u) = exp(u)"""
    return tf.exp(u)

def f(v):
    """f(v) = sin(v)"""
    return tf.sin(v)

# ── Evaluation point ──────────────────────────────────────────────────────────
x_val = 1.0
x = tf.Variable(x_val, dtype=tf.float64)

# ── FORWARD PASS ──────────────────────────────────────────────────────────────
# The tape records every operation in order as it is executed
with tf.GradientTape() as tape:
    u = h(x)          # step 1: h(x) = x² + 1
    v = g(u)          # step 2: g(u) = exp(u)
    y = f(v)          # step 3: f(v) = sin(v)

print("=" * 50)
print("  FORWARD PASS  (x → u → v → y)")
print("=" * 50)
print(f"  x          = {x.numpy():.6f}")
print(f"  u = h(x)   = x²+1        = {u.numpy():.6f}")
print(f"  v = g(u)   = exp(u)      = {v.numpy():.6f}")
print(f"  y = f(v)   = sin(v)      = {y.numpy():.6f}")

# ── BACKWARD PASS via AAD ─────────────────────────────────────────────────────
dy_dx_aad = tape.gradient(y, x)

print("\n" + "=" * 50)
print("  BACKWARD PASS  (y → v → u → x)")
print("=" * 50)

# Local derivatives computed analytically (for comparison)
x_np  = x_val
u_np  = x_np**2 + 1.0
v_np  = np.exp(u_np)

dh_dx = 2.0 * x_np          # h'(x) = 2x
dg_du = np.exp(u_np)         # g'(u) = exp(u)
df_dv = np.cos(v_np)         # f'(v) = cos(v)

print(f"  f'(v) = cos(v)    = {df_dv:.6f}   ← multiplier at step 3")
print(f"  g'(u) = exp(u)    = {dg_du:.6f}   ← multiplier at step 2")
print(f"  h'(x) = 2x        = {dh_dx:.6f}   ← multiplier at step 1")

dy_dx_analytical = df_dv * dg_du * dh_dx

print(f"\n  dy/dx (chain rule) = f'·g'·h' = {dy_dx_analytical:.6f}")
print(f"  dy/dx (AAD)                   = {dy_dx_aad.numpy():.6f}")
print(f"  Error                         = {abs(dy_dx_aad.numpy() - dy_dx_analytical):.2e}")



  FORWARD PASS  (x → u → v → y)
  x          = 1.000000
  u = h(x)   = x²+1        = 2.000000
  v = g(u)   = exp(u)      = 7.389056
  y = f(v)   = sin(v)      = 0.893855

  BACKWARD PASS  (y → v → u → x)
  f'(v) = cos(v)    = 0.448356   ← multiplier at step 3
  g'(u) = exp(u)    = 7.389056   ← multiplier at step 2
  h'(x) = 2x        = 2.000000   ← multiplier at step 1

  dy/dx (chain rule) = f'·g'·h' = 6.625859
  dy/dx (AAD)                   = 6.625859
  Error                         = 0.00e+00


## Important Notes on AAD with TensorFlow

1. **GradientTape**: Must enclose all operations we want to differentiate
2. **Trackable variables**: Only `tf.Variable` and some `tf.Tensor` are automatically tracked
3. **Persistent tape**: If we need to compute multiple gradients, use `persistent=True`
4. **Efficiency**: AAD is computationally efficient even for very complex functions

## Application to Black-Scholes

In the following examples, we will apply these techniques to Black-Scholes option pricing, automatically calculating the **Greeks** (Delta, Gamma, Vega, Theta, Rho) using both analytical formulas and Monte Carlo simulations.


---

# Application: Black-Scholes Pricing with AAD

Now we apply AAD techniques to option pricing using the `tensorquant` framework. We will see how to automatically calculate the Greeks (sensitivities) of an option.


## Black-Scholes Model Overview

The **Black-Scholes model** is a mathematical model for pricing European-style options. The model assumes that the price of the underlying asset follows a geometric Brownian motion with constant drift and volatility.

The Black-Scholes formula for the price of a European call option is given by:

$$
C(S, t) = S_0 N(d_1) - X e^{-r(T-t)} N(d_2)
$$

where:
- $ S_0 $ is the current price of the asset.
- $ X $ is the strike price.
- $ T $ is the time to expiration.
- $ r $ is the risk-free interest rate.
- $ N(d) $ is the cumulative distribution function of the standard normal distribution.
- $ d_1 $ and $ d_2 $ are given by:

$$
d_1 = \frac{\ln(S_0/X) + (r + \frac{\sigma^2}{2}) (T-t)}{\sigma \sqrt{T-t}}
$$

$$
d_2 = d_1 - \sigma \sqrt{T-t}
$$

### Assumptions of the Model:
1. The price of the underlying asset follows a lognormal distribution.
2. There are no transaction costs or taxes.
3. The risk-free interest rate is constant.
4. The volatility of the asset is constant.
5. The options are European-style, meaning they can only be exercised at expiration.

---

## Key Sensitivities (Greeks)

In the Black-Scholes framework, the price of an option is sensitive to various factors, commonly referred to as the **Greeks**. The most important Greeks are:

- **Delta (Δ)**: Measures the sensitivity of the option price to small changes in the price of the underlying asset. Mathematically, it is the partial derivative of the option price with respect to the asset price.

  $$
  \Delta = \frac{\partial C}{\partial S_0}
  $$

- **Gamma (Γ)**: Measures the sensitivity of Delta to small changes in the price of the underlying asset.

  $$
  \Gamma = \frac{\partial^2 C}{\partial S_0^2}
  $$

- **Theta (Θ)**: Measures the sensitivity of the option price to the passage of time. It is the partial derivative of the option price with respect to time.

  $$
  \Theta = \frac{\partial C}{\partial t}
  $$

- **Vega (ν)**: Measures the sensitivity of the option price to changes in volatility of the underlying asset.

  $$
  \nu = \frac{\partial C}{\partial \sigma}
  $$

- **Rho (ρ)**: Measures the sensitivity of the option price to changes in the risk-free interest rate.

  $$
  \rho = \frac{\partial C}{\partial r}
  $$


In [8]:
ref_date = tq.Settings.evaluation_date
calendar = tq.TARGET()
daycounter = tq.DayCounter(tq.DayCounterConvention.Actual360)

disc_curve = tq.FlatCurve(ref_date, 0.03, tq.DayCounterConvention.Actual360)
flat_vol = tq.BlackConstantVolatility(ref_date, 0.18)
spot_price = 6100

In [9]:
market = {
    "IR:EUR:ESTR:SPOT":    disc_curve,   
    "EQ:EUR:DEFAULT:SPOT": 6100.0,       # Spot dell'underlying
    "EQ:EUR:DEFAULT:VOL":  flat_vol,     # Superficie di volatilità
}

# 3. Creiamo il MarketEnvironment
# - Valida la struttura e i tipi del market dict
# - Valida la struttura del market_map (default da utils.py se non fornito)
# - Cross-valida che tutte le chiavi in market_map siano presenti in market
market_env = tq.MarketEnvironment(market)

T = 2.
maturity = calendar.advance(ref_date, T, tq.TimeUnit.Years, tq.BusinessDayConvention.ModifiedFollowing)
option = tq.VanillaOption(
                        tq.Currency.EUR,
                        ref_date,
                        maturity,
                        tq.OptionType.Call,
                        strike=6107
                        )

In [10]:
pricer = tq.BlackScholesPricer()
pricer.price(option, market_env, autodiff=True)

In [11]:
pricer.tape.gradient(option.price, [pricer._s, pricer._sigma, pricer._r, pricer._t, ])

[<tf.Tensor: shape=(), dtype=float32, numpy=0.6414201259613037>,
 <tf.Tensor: shape=(), dtype=float32, numpy=3227.3994140625>,
 <tf.Tensor: shape=(), dtype=float32, numpy=6253.880859375>,
 <tf.Tensor: shape=(), dtype=float32, numpy=239.68743896484375>]

In [12]:
option.price

<tf.Tensor: shape=(), dtype=float32, numpy=794.265869140625>

## Monte Carlo Simulation for Option Pricing

Monte Carlo simulations are used to model the behavior of financial instruments under uncertainty. In the context of option pricing, Monte Carlo methods involve simulating numerous paths for the price of the underlying asset using random samples from a probability distribution (typically normal distribution).

### Steps in a Monte Carlo Simulation for Option Pricing:
1. Simulate multiple paths for the underlying asset price using a stochastic process like **Geometric Brownian Motion (GBM)**:
   
   $$
   S_t = S_0 e^{(r - \frac{\sigma^2}{2})t + \sigma W_t}
   $$
   
   where $ W_t $ is a Wiener process (Brownian motion).

2. Calculate the payoff for each simulated path at option maturity. For a call option, the payoff is:
   
   $$
   \max(S_T - X, 0)
   $$
   
3. Discount the average payoff back to the present to get the option price:
   
   $$
   C = e^{-rT} \frac{1}{n} \sum_{i=1}^{n} \max(S_T^i - X, 0)
   $$

By simulating a large number of paths, the Monte Carlo approach provides an approximation for the option price, particularly useful for options with complex features that make closed-form solutions difficult.

In [13]:
model  = tq.GeometricBrownianMotion(mu=disc_curve.rate, sigma=flat_vol.volatility().numpy(), x0=spot_price)

end_date = calendar.advance(ref_date, T, tq.TimeUnit.Years, tq.BusinessDayConvention.ModifiedFollowing)
schedule_gen = tq.ScheduleGenerator(calendar, tq.BusinessDayConvention.ModifiedFollowing)
date_grid = schedule_gen.generate(ref_date, end_date, 1, tq.TimeUnit.Weeks)
time_grid = [daycounter.year_fraction(ref_date, d) for d in date_grid]

n_path = 100000
z   = tf.random.normal((n_path, len(time_grid)), seed=12, dtype=tf.dtypes.float64)

with tf.GradientTape() as tape:
    s_t = model.evolve(time_grid, z)
    payoff = tf.math.maximum(s_t[:, -1] - option.strike.numpy(), 0)
    p_mc = disc_curve.discount(T) * tf.reduce_mean(payoff)

sensy = tape.gradient(p_mc, [model._x0, model._mu, model._sigma])
print("Black AAD: ")
print("Price: ", p_mc.numpy())
print("*"*30)
print(f"Delta: {sensy[0].numpy()}")
print(f"Rho: {sensy[1].numpy()}")
print(f"Vega: {sensy[2].numpy()}")
print("*"*30)

Black AAD: 
Price:  805.0316099480398
******************************
Delta: 0.6434565131094977
Rho: 7981.00561760147
Vega: 3292.8869402160826
******************************


# Relation to Artificial Intelligence

In [14]:
# Simple backpropagation example with a tiny neural network

import tensorflow as tf

print("\n=== Backpropagation example: 2-layer neural network ===\n")

# Single input x and target y_target
x = tf.constant([[1.0]], dtype=tf.float32)  # shape (1, 1)
y_target = tf.constant([[2.0]], dtype=tf.float32)  # we want the network to output 2.0

# Define a tiny network: x -> Dense(1, relu) -> Dense(1)
layer1 = tf.keras.layers.Dense(1, activation="relu")
layer2 = tf.keras.layers.Dense(1)

# Forward + backward with GradientTape
with tf.GradientTape() as tape:
    tape.watch(layer1.trainable_variables + layer2.trainable_variables)
    h = layer1(x)           # first layer
    y_pred = layer2(h)      # second layer
    loss = tf.reduce_mean((y_pred - y_target)**2)  # MSE loss

# Backpropagation: compute gradients of the loss w.r.t. parameters
grads = tape.gradient(loss, layer1.trainable_variables + layer2.trainable_variables)

print("Input x:", x.numpy())
print("Target y:", y_target.numpy())
print("Predicted y:", y_pred.numpy())
print("Loss:", loss.numpy())
print("\nGradients (backprop / reverse-mode AD):")
for var, g in zip(layer1.trainable_variables + layer2.trainable_variables, grads):
    print(f"dLoss/d{var.name} =\n", g.numpy())



=== Backpropagation example: 2-layer neural network ===

Input x: [[1.]]
Target y: [[2.]]
Predicted y: [[0.]]
Loss: 4.0

Gradients (backprop / reverse-mode AD):
dLoss/dkernel =
 [[0.]]
dLoss/dbias =
 [-0.]
dLoss/dkernel =
 [[0.]]
dLoss/dbias =
 [-4.]


## Manual training example: learning the sum y = x1 + x2

In this example we use backpropagation (reverse-mode AD) to train a tiny neural network to approximate the function:

$$y = x_1 + x_2$$

We will:
- create random training data $(x_1, x_2, y)$ with $y = x_1 + x_2$,
- define a 2-input / 1-output linear layer (a single neuron),
- run a manual training loop using `GradientTape`,
- see that the learned weights converge close to `[1, 1]` (and bias close to `0`).


In [15]:
print("\n=== Manual training: learn y = x1 + x2 with a linear neuron ===\n")

# 1. Create synthetic training data
n_samples = 1000
rng = np.random.default_rng(seed=42)

x1 = rng.uniform(-1.0, 1.0, size=(n_samples, 1)).astype(np.float32)
x2 = rng.uniform(-1.0, 1.0, size=(n_samples, 1)).astype(np.float32)
X = np.concatenate([x1, x2], axis=1)              # shape (n_samples, 2)
y = (x1 + x2).astype(np.float32)                  # shape (n_samples, 1)

X_tf = tf.constant(X, dtype=tf.float32)
y_tf = tf.constant(y, dtype=tf.float32)

# 2. Define a 2->1 linear layer: y_hat = w1*x1 + w2*x2 + b
model = tf.keras.layers.Dense(1, use_bias=True, input_shape=(2,))

# 3. Manual training loop
learning_rate = 0.1
epochs = 200

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        y_pred = model(X_tf)
        loss = tf.reduce_mean((y_pred - y_tf) ** 2)  # MSE

    # Compute gradients of loss w.r.t. parameters (backprop)
    grads = tape.gradient(loss, model.trainable_variables)

    # Manual gradient descent update
    for var, g in zip(model.trainable_variables, grads):
        var.assign_sub(learning_rate * g)

    if (epoch + 1) % 40 == 0:
        w, b = model.get_weights()
        print(f"Epoch {epoch+1:3d} | Loss = {loss.numpy():.6f}")
        print("  Weights (w1, w2):", w.ravel())
        print("  Bias:", b, "\n")

# Final parameters
w_final, b_final = model.get_weights()
print("Final learned weights (w1, w2):", w_final.ravel())
print("Final learned bias:", b_final)

# Quick test on a new point
x_test = tf.constant([[0.3, -0.2]], dtype=tf.float32)
y_true = 0.3 + (-0.2)
y_pred_test = model(x_test).numpy()[0, 0]
print("\nTest point x = [0.3, -0.2]")
print("True y =", y_true)
print("Predicted y =", y_pred_test)



=== Manual training: learn y = x1 + x2 with a linear neuron ===

Epoch  40 | Loss = 0.003315
  Weights (w1, w2): [0.9570053  0.91795915]
  Bias: [0.00172351] 

Epoch  80 | Loss = 0.000013
  Weights (w1, w2): [0.9974493  0.99486667]
  Bias: [0.00010925] 

Epoch 120 | Loss = 0.000000
  Weights (w1, w2): [0.9998488  0.99967873]
  Bias: [6.904511e-06] 



c:\Documenti\tqenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 160 | Loss = 0.000000
  Weights (w1, w2): [0.9999912 0.9999799]
  Bias: [4.3628475e-07] 

Epoch 200 | Loss = 0.000000
  Weights (w1, w2): [0.99999946 0.99999875]
  Bias: [3.091457e-08] 

Final learned weights (w1, w2): [0.99999946 0.99999875]
Final learned bias: [3.091457e-08]

Test point x = [0.3, -0.2]
True y = 0.09999999999999998
Predicted y = 0.10000014
